In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [3]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)

-43.89740030179307


In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)

-42.58225552052512


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)

-41.21967768602254


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)

-41.713397841885595


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)

-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [10]:

# Helper: safe log — returns -inf for zero-probability events
NEG_INF = float('-inf')

def safe_log(x):
    """Return log(x), or -inf if x == 0 (impossible event)."""
    return math.log(x) if x > 0 else NEG_INF

n_states = len(states)          # 5 hidden states
seq_len  = len(query_sequence)  # 26 nucleotides

# viterbi_value_matrix[state][t]  -> best log-prob of reaching 'state' at position t
# viterbi_trace_matrix[state][t]  -> index of predecessor state that gave that best prob
viterbi_value_matrix = np.full((n_states, seq_len), NEG_INF)
viterbi_trace_matrix = np.zeros((n_states, seq_len), dtype=int)

# Initialization: column 0 (first observed nucleotide)
# v[j][0] = log(P(s -> j)) + log(P(j emits obs[0])
# Trace matrix col 0: each state points to itself as per the

start_idx = states['s']
first_obs  = emission_nuc_codes[query_sequence[0]]

for j in range(n_states):
    trans = state_transition_prob[start_idx][j]
    emis  = emission_probs[j][first_obs]
    if trans > 0 and emis > 0:
        viterbi_value_matrix[j][0] = safe_log(trans) + safe_log(emis)
    # trace col 0 stores each state's own index
    viterbi_trace_matrix[j][0] = j

print(f"Matrices initialized. Shape: {viterbi_value_matrix.shape}")
print("\nviterbi_value_matrix after initialization (column 0 only):")
for name, idx in states.items():
    v = viterbi_value_matrix[idx][0]
    print(f"  {name} : {'  -inf' if v == NEG_INF else f'{v:.3f}'}")

print("\nviterbi_trace_matrix column 0 (predecessor state indices):")
for name, idx in states.items():
    print(f"  {name} : {viterbi_trace_matrix[idx][0]}")

Matrices initialized. Shape: (5, 26)

viterbi_value_matrix after initialization (column 0 only):
  s :   -inf
  E : -1.386
  5 :   -inf
  I :   -inf
  e :   -inf

viterbi_trace_matrix column 0 (predecessor state indices):
  s : 0
  E : 1
  5 : 2
  I : 3
  e : 4


### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND**

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [11]:
def calculate_prob_for_a_node(prev_col, curr_state_idx, obs_nuc_code):
    """
    Compute the best (maximum) log-probability for a single Viterbi cell
    and identify which predecessor state produced that maximum.

    Parameters
    ----------
    prev_col       : 1-D array, shape (n_states,)
                     Log-probabilities from the PREVIOUS time step (t-1).
    curr_state_idx : int
                     Index of the current hidden state (the row being filled).
    obs_nuc_code   : int  (0=A, 1=C, 2=G, 3=T)
                     Integer code of the observed nucleotide at time t.

    Returns
    -------
    best_val  : float  – maximum log-probability (-inf if state is unreachable)
    best_prev : int    – index of the predecessor state giving best_val
    """
    # If this state cannot emit the observed nucleotide, the cell is -inf
    emis = emission_probs[curr_state_idx][obs_nuc_code]
    if emis == 0:
        return NEG_INF, 0

    log_emis  = safe_log(emis)
    best_val  = NEG_INF
    best_prev = 0

    # Try every possible predecessor state
    for prev_state in range(n_states):
        trans = state_transition_prob[prev_state][curr_state_idx]
        # Skip if transition is impossible or predecessor was itself unreachable
        if trans == 0 or prev_col[prev_state] == NEG_INF:
            continue
        candidate = prev_col[prev_state] + safe_log(trans) + log_emis
        if candidate > best_val:
            best_val  = candidate
            best_prev = prev_state

    return best_val, best_prev

In [12]:

# Fill the matrices for positions t = 1 … seq_len-1
for t in range(1, seq_len):
    obs_code = emission_nuc_codes[query_sequence[t]]
    for j in range(n_states):
        val, best_prev = calculate_prob_for_a_node(
            viterbi_value_matrix[:, t - 1],   # previous column
            j,                                 # current state row
            obs_code                           # observed nucleotide code
        )
        viterbi_value_matrix[j][t] = val
        viterbi_trace_matrix[j][t] = best_prev

print("Viterbi matrix filled.")
print()
print("Final Viterbi Value Matrix (rows=states, cols=sequence positions):")
print("    ", "  ".join(f"{c:6s}" for c in query_sequence))
for state_name, idx in states.items():
    row = []
    for v in viterbi_value_matrix[idx]:
        row.append(f"{v:7.3f}" if v != NEG_INF else "   -inf")
    print(f"{state_name}: {'  '.join(row)}")

Viterbi matrix filled.

Final Viterbi Value Matrix (rows=states, cols=sequence positions):
     C       T       T       C       A       T       G       T       G       A       A       A       G       C       A       G       A       C       G       T       A       A       G       T       C       A     
s:    -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf     -inf
E:  -1.386   -2.878   -4.370   -5.861   -7.353   -8.845  -10.336  -11.828  -13.320  -14.811  -16.303  -17.794  -19.286  -20.778  -22.269  -23.761  -25.253  -26.744  -28.236  -29.728  -31.219  -32.711  -34.203  -35.694  -37.186  -38.678
5:    -inf     -inf     -inf     -inf  -11.160     -inf  -11.198     -inf  -14.182  -18.618  -20.110  -21.601  -20.148     -inf  -26.076  -24.623  -29.059     -inf  -29.098     -inf  -35.026  -36.518  -35.065     -inf     -in

In [13]:
def traceback_state_path(viterbi_value_matrix, viterbi_trace_matrix, id2state, seq_len):
    """
    Trace back the most probable hidden state sequence.

    Steps
    -----
    1. Find argmax in the LAST column of viterbi_value_matrix —
       this gives the state of the final nucleotide.
    2. Follow viterbi_trace_matrix backwards from t=seq_len-1
       down to t=1 to recover the full path.
    3. Reverse the collected indices for forward-order output.

    Parameters
    ----------
    viterbi_value_matrix : ndarray (n_states x seq_len)
    viterbi_trace_matrix : ndarray (n_states x seq_len)
    id2state             : dict  mapping int -> state character
    seq_len              : int

    Returns
    -------
    state_path   : str    most probable hidden state sequence
    max_log_prob : float  log-probability of that optimal path
    """
    # Step 1: best state at the last position
    last_col     = viterbi_value_matrix[:, seq_len - 1]
    best_last    = int(np.argmax(last_col))
    max_log_prob = last_col[best_last]

    # Step 2: walk backwards through the trace matrix
    path_indices = [best_last]
    current      = best_last
    for t in range(seq_len - 1, 0, -1):
        current = viterbi_trace_matrix[current][t]
        path_indices.append(current)

    # Step 3: reverse to get forward-order path
    path_indices.reverse()
    state_path = ''.join(id2state[i] for i in path_indices)

    return state_path, max_log_prob


state_path, max_log_prob = traceback_state_path(
    viterbi_value_matrix, viterbi_trace_matrix, id2state, seq_len
)

print("=== Viterbi Traceback Result ===")
print(f"Query sequence : {query_sequence}")
print(f"Best state path: {state_path}")
print(f"Max log-prob   : {max_log_prob:.5f}")
print()
print("Interpretation:")
print("The Viterbi algorithm finds the ALL-EXON path as most probable.")
print("The sequence does not contain a strong enough GT donor signal")
print("to overcome the E->5 transition penalty (log 0.1 = -2.303).")

=== Viterbi Traceback Result ===
Query sequence : CTTCATGTGAAAGCAGACGTAAGTCA
Best state path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Max log-prob   : -38.67767

Interpretation:
The Viterbi algorithm finds the ALL-EXON path as most probable.
The sequence does not contain a strong enough GT donor signal
to overcome the E->5 transition penalty (log 0.1 = -2.303).
